# Day 17 — Time Series Basics: Hands-On Activity

This notebook walks through the **complete time series analysis workflow** on a
synthetic daily-sales dataset:

1. Load the time series dataset
2. Convert and verify the date column
3. Extract date features
4. Aggregate to monthly totals
5. Identify the trend with a moving average
6. Analyze weekly seasonality
7. Calculate a growth KPI
8. Visualize the trend
9. Document the findings

**Requirements:** `numpy`, `pandas`, `matplotlib`

> The dataset is **synthetic** — generated with a fixed random seed so results are
> reproducible. No real or personal data is used.

## Step 1 — Load the time series dataset

**Objective:** Create a daily sales time series spanning several months.

We build 120 days of sales from three components:

- **trend** — a straight line rising from 1000 to 1600 (the upward growth),
- **weekly** — a sine wave with a 7-day period (the weekly seasonal pattern),
- **noise** — small random fluctuations (real-world randomness).

Adding them together gives a realistic series with trend + seasonality + noise.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Generate 120 days of sales with an upward trend and weekly seasonality
np.random.seed(42)
dates = pd.date_range("2025-01-01", periods=120, freq="D")
trend = np.linspace(1000, 1600, 120)                    # rising trend
weekly = 150 * np.sin(np.arange(120) * (2 * np.pi / 7)) # weekly pattern
noise = np.random.normal(0, 50, 120)                    # random noise
sales = (trend + weekly + noise).round(0)

df = pd.DataFrame({"Date": dates, "Sales": sales})
print(df.head())

## Step 2 — Convert and verify the date column

Ensure the `Date` column is a true `datetime` type so date methods are available.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
print("Date column type:", df["Date"].dtype)

**Expected output:**

```
Date column type: datetime64[ns]
```

## Step 3 — Extract date features

Pull the month name and weekday name out of the datetime column — these power the
monthly aggregation and the weekly-seasonality analysis later.

In [ ]:
df["Month"] = df["Date"].dt.month_name()
df["DayName"] = df["Date"].dt.day_name()
print(df[["Date", "Sales", "Month", "DayName"]].head())

## Step 4 — Aggregate to monthly totals

Group by month number and sum the sales. The monthly totals should **rise from
January to April**, reflecting the upward trend.

In [ ]:
df["MonthNum"] = df["Date"].dt.month
monthly = df.groupby("MonthNum")["Sales"].sum()
print(monthly)

## Step 5 — Identify the trend with a moving average

A **7-day moving average** spans a full week, smoothing the weekly cycle so the
underlying trend stands out. It should rise steadily, confirming the upward trend.

In [ ]:
df["MovingAvg_7"] = df["Sales"].rolling(window=7).mean()
print(df[["Date", "Sales", "MovingAvg_7"]].tail())

## Step 6 — Analyze weekly seasonality

Average sales per weekday. The averages should **differ by weekday**, revealing
the weekly seasonal pattern baked in by the sine wave.

In [ ]:
weekday_avg = df.groupby("DayName")["Sales"].mean().round(0)
print(weekday_avg)

## Step 7 — Calculate a growth KPI

Express the change from the first month to the last as a percentage — a single
headline KPI for the growth.

In [ ]:
growth = (monthly.iloc[-1] - monthly.iloc[0]) / monthly.iloc[0] * 100
print("Growth from first to last month: {:.1f}%".format(growth))

## Step 8 — Visualize the trend

Plot the noisy daily sales together with the smooth 7-day moving average. The
chart should show **noisy daily sales with a clear upward trend** traced by the
moving average.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["Sales"], alpha=0.4, label="Daily Sales")
plt.plot(df["Date"], df["MovingAvg_7"], color="red", linewidth=2,
         label="7-Day Moving Average")
plt.title("Daily Sales with Trend")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

## Step 9 — Document the findings

- Sales show a **clear upward trend** over the four months, confirmed by the
  rising moving average and a strong month-over-month increase.
- A **weekly seasonal pattern** is present, with certain weekdays consistently
  higher than others.
- The underlying growth, separated from noise by the moving average, is **steady
  and positive**.

**Recommendation:** continue the current strategy driving the growth, and align
operations with the weekly demand pattern.

---

Upon completion, the full time series workflow has been performed: **loading, date
conversion, feature extraction, aggregation, trend identification, seasonality
analysis, KPI calculation, and visualization.**